# Chapter 5
## The Simple Model of Neurons in Rodent Brains
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter05.ipynb)

## About this chapter

This chapter compares the RTM excitatory-neuron model with Wang-Buzsaki
(WB) and Erisir inhibitory-neuron models. All are conductance-based, but
their maximal conductances, reversal potentials, and gating kinetics
produce distinct spike waveforms and recovery speeds.

The models share sodium, potassium, and leak currents, while their rate
laws encode cell-type-specific kinetics. In these implementations, sodium
activation $m$ is set instantaneously to its steady-state value; $h$ and
$n$ remain dynamic. The two Erisir traces differ in the power used for the
potassium gate, made directly visible here via the `n_power` kwarg.

The common current-balance form is

$$
C\frac{dV}{dt}=I_{\mathrm{ext}}-g_{\mathrm{Na}}m_\infty(V)^3h(V-E_{\mathrm{Na}})
-g_{\mathrm{K}}n^p(V-E_{\mathrm{K}})-g_{\mathrm{L}}(V-E_{\mathrm{L}}).
$$

Here $V$ is membrane voltage, $t$ is time, $C$ is capacitance,
$I_{\mathrm{ext}}$ is applied current, $g_{\mathrm{Na}}$, $g_{\mathrm{K}}$,
and $g_{\mathrm{L}}$ are maximal conductances, $E_{\mathrm{Na}}$,
$E_{\mathrm{K}}$, and $E_{\mathrm{L}}$ are reversal potentials, $m_\infty$
is the instantaneous sodium-activation value, $h$ is sodium inactivation,
$n$ is potassium activation, and $p$ is the potassium-gate exponent
selected by the model. Dynamic gates obey
$dx/dt=\alpha_x(V)(1-x)-\beta_x(V)x$.

See [`README.md`](chapter05.md)
for the full guide, including suggested order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact
from mnd.core import alpha_h, alpha_m, alpha_n, beta_h, beta_m, beta_n, h_inf, m_inf, n_inf

## Reduced Traub-Miles (RTM) Model

`m` is not its own state -- it is held at instantaneous equilibrium
`m_inf(v)` at every step, same as `make_figure.m`.

In [ ]:
def simulate_rtm_voltage_trace(c=1, g_k=80, g_na=100, g_l=0.1,
                                v_k=-100, v_na=50, v_l=-67,
                                i_ext=1.5, t_final=100, dt=0.01):
    def derivative(x0, t):
        v, n, h = x0
        m = m_inf(v)
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l))
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dn, dh]

    v0 = -70.0
    x0 = [v0, n_inf(v0), h_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0]


def plot_voltage_trace(t, v):
    plt.figure(figsize=(7, 3))
    plt.plot(t, v, lw=2, c="k")
    plt.xlim(min(t), max(t))
    plt.ylim(-100, 50)
    plt.xlabel("time [ms]")
    plt.ylabel("v [mV]")
    plt.yticks(range(-100, 100, 50))
    plt.tight_layout()
    plt.show()

In [ ]:
plot_voltage_trace(*simulate_rtm_voltage_trace())

In [ ]:
interact(lambda i_ext=1.5: plot_voltage_trace(*simulate_rtm_voltage_trace(i_ext=i_ext)),
         i_ext=(0.0, 5.0, 0.1));

### RTM Gating Variables

Steady-state activation/inactivation and time constants for the RTM gates.

In [ ]:
def simulate_rtm_gating_variables(v=None):
    if v is None:
        v = np.arange(-100, 50, 0.01)
    tau_m = 1.0 / (alpha_m(v) + beta_m(v))
    tau_h = 1.0 / (alpha_h(v) + beta_h(v))
    tau_n = 1.0 / (alpha_n(v) + beta_n(v))
    return v, m_inf(v), h_inf(v), n_inf(v), tau_m, tau_h, tau_n


def plot_rtm_gating_variables(v, m_inf_v, h_inf_v, n_inf_v, tau_m, tau_h, tau_n):
    fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(7, 7))

    ax[0][0].plot(v, m_inf_v, lw=2, c="k")
    ax[1][0].plot(v, h_inf_v, lw=2, c="k")
    ax[2][0].plot(v, n_inf_v, lw=2, c="k")

    ax[0][1].plot(v, tau_m, lw=2, c="k")
    ax[1][1].plot(v, tau_h, lw=2, c="k")
    ax[2][1].plot(v, tau_n, lw=2, c="k")

    ax[0][0].set_ylabel(r"$m_{\infty} (v)$")
    ax[1][0].set_ylabel(r"$h_{\infty} (v)$")
    ax[2][0].set_ylabel(r"$n_{\infty} (v)$")

    ax[0][1].set_ylabel(r"$\tau_m [ms]$")
    ax[1][1].set_ylabel(r"$\tau_h [ms]$")
    ax[2][1].set_ylabel(r"$\tau_n [ms]$")

    ax[2][0].set_xlabel("v [mV]", fontsize=14)
    ax[2][1].set_xlabel("v [mV]", fontsize=14)

    for i in range(3):
        for j in range(2):
            ax[i][j].set_xlim(min(v), max(v))
            ax[i][j].set_ylim([0, 1.05])
    ax[1][1].set_ylim([0, 20])
    ax[2][1].set_ylim([0, 20])

    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_gating_variables(*simulate_rtm_gating_variables())

## Wang-Buzsaki (WB) Model

2003-style parameterization (implicit \(\phi=1\)).

In [ ]:
def simulate_wb_voltage_trace(c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
                               v_k=-90.0, v_na=55.0, v_l=-65.0,
                               i_ext=0.75, t_final=100.0, dt=0.01):
    def alpha_h(v):
        return 0.35 * exp(-(v + 58.0) / 20.0)

    def alpha_m(v):
        return 0.1 * (v + 35.0) / (1.0 - exp(-0.1 * (v + 35.0)))

    def alpha_n(v):
        return -0.05 * (v + 34.0) / (exp(-0.1 * (v + 34.0)) - 1.0)

    def beta_h(v):
        return 5.0 / (exp(-0.1 * (v + 28.0)) + 1.0)

    def beta_m(v):
        return 4.0 * exp(-(v + 60.0) / 18.0)

    def beta_n(v):
        return 0.625 * exp(-(v + 44.0) / 80.0)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def h_inf(v):
        return alpha_h(v) / (alpha_h(v) + beta_h(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def derivative(x0, t):
        v, h, n = x0
        i_na = g_na * m_inf(v) ** 3 * h * (v - v_na)
        i_l = g_l * (v - v_l)
        i_k = g_k * n ** 4 * (v - v_k)
        dv = -i_na - i_k - i_l + i_ext
        dh = alpha_h(v) * (1 - h) - beta_h(v) * h
        dn = alpha_n(v) * (1 - n) - beta_n(v) * n
        return [dv, dh, dn]

    v0 = -63.0
    x0 = [v0, h_inf(v0), n_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0]

In [ ]:
plot_voltage_trace(*simulate_wb_voltage_trace())

In [ ]:
interact(lambda i_ext=0.75: plot_voltage_trace(*simulate_wb_voltage_trace(i_ext=i_ext)),
         i_ext=(0.0, 3.0, 0.05));

### WB Model, 1996 Parameterization

Wang & Buzsaki (1996), "Gamma Oscillation by Synaptic Inhibition in a
Hippocampal Interneuronal Network Model" -- distinct rate coefficients from
the 2003-style version above, plus an explicit time-scaling factor
\(\phi=5\).

In [ ]:
def simulate_wb_voltage_trace_1996(c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
                                    v_k=-90.0, v_na=55.0, v_l=-65.0,
                                    i_ext=0.75, t_final=100.0, dt=0.01, phi=5.0):
    def alpha_h(v):
        return 0.07 * exp(-(v + 58.0) / 20.0)

    def alpha_m(v):
        return -0.1 * (v + 35.0) / (exp(-0.1 * (v + 35.0)) - 1.0)

    def alpha_n(v):
        return -0.01 * (v + 34.0) / (exp(-0.1 * (v + 34.0)) - 1.0)

    def beta_h(v):
        return 1.0 / (exp(-0.1 * (v + 28.0)) + 1.0)

    def beta_m(v):
        return 4.0 * exp(-(v + 60.0) / 18.0)

    def beta_n(v):
        return 0.125 * exp(-(v + 44.0) / 80.0)

    def h_inf(v):
        return alpha_h(v) / (alpha_h(v) + beta_h(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def derivative(x0, t):
        v, h, n = x0
        m = alpha_m(v) / (alpha_m(v) + beta_m(v))
        i_na = g_na * m ** 3 * h * (v - v_na)
        i_l = g_l * (v - v_l)
        i_k = g_k * n ** 4 * (v - v_k)
        dv = -i_na - i_k - i_l + i_ext
        dh = phi * (alpha_h(v) * (1 - h) - beta_h(v) * h)
        dn = phi * (alpha_n(v) * (1 - n) - beta_n(v) * n)
        return [dv, dh, dn]

    v0 = -63.0
    x0 = [v0, h_inf(v0), n_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0]

In [ ]:
plot_voltage_trace(*simulate_wb_voltage_trace_1996())

## Erisir et al. Model

`n_power` selects between the two parameterizations used in the book:
\(n^2\) or \(n^4\) for the potassium term.

In [ ]:
def simulate_erisir_voltage_trace(c=1, g_k=224.0, g_na=112, g_l=0.5,
                                   v_k=-90.0, v_na=60, v_l=-70,
                                   i_ext=7.0, t_final=100, dt=0.01, n_power=2):
    def alpha_h(v):
        return 0.0035 / exp(v / 24.186)

    def alpha_m(v):
        return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)

    def alpha_n(v):
        return (95 - v) / (exp((95 - v) / 11.8) - 1)

    def beta_h(v):
        return -0.017 * (v + 51.25) / (exp(-(v + 51.25) / 5.2) - 1)

    def beta_m(v):
        return 1.2262 / exp(v / 42.248)

    def beta_n(v):
        return 0.025 / exp(v / 22.222)

    def h_inf(v):
        return alpha_h(v) / (alpha_h(v) + beta_h(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def derivative(x0, t):
        v, n, h = x0
        m = alpha_m(v) / (alpha_m(v) + beta_m(v))
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** n_power * (v - v_k) - g_l * (v - v_l))
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dn, dh]

    v0 = -70.0
    x0 = [v0, n_inf(v0), h_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0]

In [ ]:
plot_voltage_trace(*simulate_erisir_voltage_trace(n_power=2))

In [ ]:
plot_voltage_trace(*simulate_erisir_voltage_trace(n_power=4))

In [ ]:
interact(lambda i_ext=7.0, n_power=2: plot_voltage_trace(*simulate_erisir_voltage_trace(i_ext=i_ext, n_power=n_power)),
         i_ext=(0.0, 15.0, 0.5), n_power=[2, 4]);